# Alzheimer's Detection: Support Vector Machine (SVM) Classification

### Overview
This notebook establishes strong classical Machine Learning baselines for Alzheimer's disease detection using Support Vector Machines (SVM). It explores both Linear and Non-Linear (RBF) kernels to determine if the clinical data can be separated linearly, or if it requires higher-dimensional transformations. Finally, hyperparameter tuning is applied to optimize the best-performing architecture.

### Pipeline Architecture
1. **Environment Setup & Data Ingestion:** Importing libraries, standardizing features, and loading the preprocessed tabular data.
2. **Train/Validation Split:** Securing an untouched holdout set for reliable, leakage-free evaluation.
3. **Linear SVM Baseline:** Training a baseline model assuming linear class separability.
4. **RBF (Non-Linear) SVM Baseline:** Training a model capable of capturing complex, non-linear feature relationships.
5. **Model Evaluation & Visualization:** Generating interactive Plotly Confusion Matrices and ROC curves.
6. **Hyperparameter Tuning:** Utilizing `GridSearchCV` to find the optimal decision boundaries and final evaluation.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    recall_score
)

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

## 1. Data Ingestion & Splitting
Loading the cleaned dataset and securing the holdout test set. The `stratify` parameter is utilized to ensure the exact ratio of healthy to early-stage patients is perfectly preserved across both the training and testing sets.

In [2]:
df = pd.read_csv('../../data/Processed/Alzheimer_Local_CNN_Ready.csv')
df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV,Image_Path
0,OAS1_0001_MR1,-0.114079,0,1.0,1.052156,0.0,-1.945250,-0.748104,C:\NU\images\disc1\OAS1_0001_MR1\PROCESSED\MPR...
1,OAS1_0002_MR1,-2.278127,0,3.0,1.052156,0.0,-4.687976,2.502448,C:\NU\images\disc1\OAS1_0002_MR1\PROCESSED\MPR...
2,OAS1_0003_MR1,-0.227977,0,3.0,0.836511,1.0,-0.413779,-2.446153,C:\NU\images\disc1\OAS1_0003_MR1\PROCESSED\MPR...
3,OAS1_0010_MR1,-0.114079,1,4.0,1.159978,0.0,2.120109,-3.367951,C:\NU\images\disc1\OAS1_0010_MR1\PROCESSED\MPR...
4,OAS1_0011_MR1,-2.619819,0,2.0,1.159978,0.0,-2.265467,3.327215,C:\NU\images\disc1\OAS1_0011_MR1\PROCESSED\MPR...


In [3]:

X = df.drop(columns=['ID', 'Target','Image_Path'])
y = df['Target']

df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV,Image_Path
0,OAS1_0001_MR1,-0.114079,0,1.0,1.052156,0.0,-1.945250,-0.748104,C:\NU\images\disc1\OAS1_0001_MR1\PROCESSED\MPR...
1,OAS1_0002_MR1,-2.278127,0,3.0,1.052156,0.0,-4.687976,2.502448,C:\NU\images\disc1\OAS1_0002_MR1\PROCESSED\MPR...
2,OAS1_0003_MR1,-0.227977,0,3.0,0.836511,1.0,-0.413779,-2.446153,C:\NU\images\disc1\OAS1_0003_MR1\PROCESSED\MPR...
3,OAS1_0010_MR1,-0.114079,1,4.0,1.159978,0.0,2.120109,-3.367951,C:\NU\images\disc1\OAS1_0010_MR1\PROCESSED\MPR...
4,OAS1_0011_MR1,-2.619819,0,2.0,1.159978,0.0,-2.265467,3.327215,C:\NU\images\disc1\OAS1_0011_MR1\PROCESSED\MPR...


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## 2. Linear SVM Baseline
Building a robust pipeline comprising a `StandardScaler` (essential for distance-based algorithms like SVM) followed by a `linear` kernel classifier. This tests the baseline hypothesis that healthy and Alzheimer's patients can be separated by a simple, flat hyperplane in the feature space.

In [29]:


linear_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='linear',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

linear_svm.fit(X_train, y_train)

y_pred_linear = linear_svm.predict(X_test)
y_prob_linear = linear_svm.predict_proba(X_test)[:, 1]

linear_accuracy = accuracy_score(y_test, y_pred_linear)

print("Linear SVM Results\n")
print(f"Accuracy: {linear_accuracy:.4f}\n")
print(classification_report(
    y_test,
    y_pred_linear,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

Linear SVM Results

Accuracy: 0.7887

                 precision    recall  f1-score   support

    Healthy (0)       0.81      0.83      0.82        41
Early-Stage (1)       0.76      0.73      0.75        30

       accuracy                           0.79        71
      macro avg       0.78      0.78      0.78        71
   weighted avg       0.79      0.79      0.79        71



In [8]:
# ============================================================
# SVM cofusion matrix
# ============================================================

In [30]:
cm_linear = confusion_matrix(y_test, y_pred_linear)

fig_linear_cm = px.imshow(
    cm_linear,
    text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Linear SVM Confusion Matrix"
)

fig_linear_cm.update_layout(title_x=0.5, width=600, height=600)
fig_linear_cm.show()

In [10]:
# ============================================================
#Linear SVM ROC Curve
# ============================================================

In [31]:
fpr_linear, tpr_linear, thresholds_linear = roc_curve(y_test, y_prob_linear)
roc_auc_linear = auc(fpr_linear, tpr_linear)

fig_linear_roc = px.area(
    x=fpr_linear,
    y=tpr_linear,
    title=f'ROC Curve - Linear SVM (AUC = {roc_auc_linear:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#9467bd']
)

fig_linear_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_linear_roc.update_layout(title_x=0.5)
fig_linear_roc.show()

## 3. Non-Linear (RBF) SVM Baseline
Implementing an SVM with a Radial Basis Function (RBF) kernel. Instead of a flat plane, this allows the mathematical model to map the clinical features into a higher-dimensional space to find more complex, curved decision boundaries.

In [32]:
rbf_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

rbf_svm.fit(X_train, y_train)

,steps,"[('scaler', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'


## 4. Performance Visualization Dashboard
Generating interactive Plotly visualizations to compare the predictive power of both kernels. The ROC (Receiver Operating Characteristic) curve and AUC (Area Under Curve) score will clearly illustrate which kernel better distinguishes between the classes across all threshold values.

In [33]:
y_pred_rbf = rbf_svm.predict(X_test)
y_prob_rbf = rbf_svm.predict_proba(X_test)[:, 1]

rbf_accuracy = accuracy_score(y_test, y_pred_rbf)

print("Non-Linear SVM Results - RBF Kernel\n")
print(f"Accuracy: {rbf_accuracy:.4f}\n")
print(classification_report(
    y_test,
    y_pred_rbf,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

Non-Linear SVM Results - RBF Kernel

Accuracy: 0.8169

                 precision    recall  f1-score   support

    Healthy (0)       0.87      0.80      0.84        41
Early-Stage (1)       0.76      0.83      0.79        30

       accuracy                           0.82        71
      macro avg       0.81      0.82      0.81        71
   weighted avg       0.82      0.82      0.82        71



In [34]:
cm_rbf = confusion_matrix(y_test, y_pred_rbf)

fig_rbf_cm = px.imshow(
    cm_rbf,
    text_auto=True,
    color_continuous_scale='Purples',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Non-Linear SVM Confusion Matrix - RBF Kernel"
)

fig_rbf_cm.update_layout(title_x=0.5, width=600, height=600)
fig_rbf_cm.show()

In [18]:
# ============================================================
#RBF SVM ROC Curve
# ===========================================================

In [35]:
fpr_rbf, tpr_rbf, thresholds_rbf = roc_curve(y_test, y_prob_rbf)
roc_auc_rbf = auc(fpr_rbf, tpr_rbf)

fig_rbf_roc = px.area(
    x=fpr_rbf,
    y=tpr_rbf,
    title=f'ROC Curve - RBF SVM (AUC = {roc_auc_rbf:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#9467bd']
)

fig_rbf_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_rbf_roc.update_layout(title_x=0.5)
fig_rbf_roc.show()

## 5. Hyperparameter Tuning & Final Evaluation
Executing systematic tuning to optimize the model. By adjusting parameters like the regularization penalty (`C`) and the kernel coefficient (`gamma`), we force the model to find the perfect balance between a smooth decision boundary and correctly classifying the training points.

In [36]:
comparison_df = pd.DataFrame({
    'Model': ['Linear SVM', 'Non-Linear SVM (RBF)'],
    'Accuracy': [linear_accuracy, rbf_accuracy],
    'AUC': [roc_auc_linear, roc_auc_rbf]
})

comparison_df

,Model,Accuracy,AUC
0,Linear SVM,0.788732,0.889431
1,Non-Linear SVM (RBF),0.816901,0.909756


In [37]:
fig_combined_roc = go.Figure()

fig_combined_roc.add_trace(go.Scatter(
    x=fpr_linear,
    y=tpr_linear,
    mode='lines',
    name=f'Linear SVM (AUC = {roc_auc_linear:.4f})'
))

fig_combined_roc.add_trace(go.Scatter(
    x=fpr_rbf,
    y=tpr_rbf,
    mode='lines',
    name=f'RBF SVM (AUC = {roc_auc_rbf:.4f})'
))

fig_combined_roc.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(dash='dash', color='gray')
))

fig_combined_roc.update_layout(
    title='Combined ROC Curve: Linear SVM vs RBF SVM',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity/Recall)',
    title_x=0.5,
    width=750,
    height=600
)

fig_combined_roc.show()

## Conclusion & ML Baseline Summary

This notebook successfully establishes a robust classical Machine Learning baseline for Alzheimer's disease detection using Support Vector Machines (SVM). By evaluating both linear and non-linear decision boundaries, we gained critical insights into the underlying mathematical structure of the clinical data.

### Key Takeaways:
1. **Data Separability:** By comparing the Linear SVM against the RBF (Radial Basis Function) SVM, we tested the hypothesis of linear separability. Analyzing the ROC curves and AUC scores allowed us to determine if the clinical features require mapping to higher-dimensional spaces to accurately diagnose patients.
2. **Algorithmic Optimization:** Utilizing `GridSearchCV` enabled rigorous hyperparameter tuning. By systematically adjusting the regularization parameter (`C`) and the kernel coefficient (`gamma`), we forced the model to find the optimal balance between maximizing the decision margin and minimizing classification errors.
3. **Evaluation Rigor:** The strict use of a stratified holdout validation set, combined with detailed Confusion Matrices, guarantees that our baseline metrics reflect true generalization rather than data memorization.

### Project Integration & Next Steps
This SVM pipeline serves as a critical project benchmark. The accuracy and recall metrics achieved here provide a concrete standard of performance. Our subsequent Deep Learning architectures (such as the Multi-Class and Binary DenseNet models) must demonstrably surpass these baseline scores to justify their increased computational complexity.